# C05 单因子到多因子（在线版）

本节使用在线真实行情 + 因子，完成单因子检验与多因子整合。


In [ ]:
START_DATE = "2021-01-01"
END_DATE = "2024-12-31"
MAX_ASSETS = 80

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 方式1（推荐课堂演示）：把教育版 license 直接粘贴到 PASSWD
PASSWD = "在这里粘贴你的教育版license"

# 方式2（不想明文写在 notebook）：注释掉上面一行，改用环境变量
# import os
# PASSWD = os.getenv("RQDATAC_LICENSE", "")

if PASSWD and ("在这里粘贴" not in PASSWD):
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


## 1) 选取在线股票池（HS300 成分）


In [ ]:
base_date = rqdatac.get_previous_trading_date(END_DATE)
asset_list = sorted(rqdatac.index_components("000300.XSHG", date=base_date))[:MAX_ASSETS]

close = rqdatac.get_price(
    asset_list,
    start_date=START_DATE,
    end_date=END_DATE,
    fields="close",
    adjust_type="post",
    expect_df=False,
)
close = close.droplevel(0, axis=1) if isinstance(close.columns, pd.MultiIndex) else close

# 使用在线财务因子：market_cap_3
market_cap = rqdatac.get_factor(asset_list, "market_cap_3", start_date=START_DATE, end_date=END_DATE, expect_df=False)

close.shape, market_cap.shape


## 2) 构造月频截面数据
- 因子：size(=-log 市值)、mom20、vol20
- 目标：下月收益


In [ ]:
month_ends = pd.DatetimeIndex(close.resample("ME").last().index)
mom20 = close.pct_change(20)
vol20 = close.pct_change().rolling(20).std()

panel_rows = []
for d in month_ends:
    if d not in close.index:
        continue
    for a in asset_list:
        c0 = close.loc[d, a]
        # 找到下一个月末作为未来收益终点
        idx = month_ends.get_loc(d)
        if idx + 1 >= len(month_ends):
            continue
        d1 = month_ends[idx + 1]
        if d1 not in close.index:
            continue
        c1 = close.loc[d1, a]
        if pd.isna(c0) or pd.isna(c1) or c0 == 0:
            continue

        panel_rows.append({
            "date": d,
            "asset": a,
            "size": -np.log(market_cap.loc[d, a]) if (d in market_cap.index and pd.notna(market_cap.loc[d, a]) and market_cap.loc[d, a] > 0) else np.nan,
            "mom20": mom20.loc[d, a] if d in mom20.index else np.nan,
            "vol20": vol20.loc[d, a] if d in vol20.index else np.nan,
            "future_ret": c1 / c0 - 1,
        })

panel = pd.DataFrame(panel_rows).dropna()
panel.head()


## 3) 单因子检验：winsorize + zscore + RankIC + Q5-Q1


In [ ]:
def winsorize(s, n=3):
    m, sd = s.mean(), s.std()
    return s.clip(m - n * sd, m + n * sd)


def zscore(s):
    return (s - s.mean()) / (s.std() + 1e-12)

pc = panel.copy()
for col in ["size", "mom20", "vol20"]:
    pc[col] = pc.groupby("date")[col].transform(winsorize)
    pc[col] = pc.groupby("date")[col].transform(zscore)

rank_ic = pc.groupby("date")[["mom20", "future_ret"]].apply(
    lambda x: x["mom20"].rank().corr(x["future_ret"].rank())
)

pc["q"] = pc.groupby("date")["mom20"].transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False))
qret = pc.groupby(["date", "q"])["future_ret"].mean().unstack()
ls = qret[4] - qret[0]
ls_nav = (1 + ls).cumprod()

rank_ic.describe().round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ls_nav.plot(ax=ax, title="C05 在线版单因子多空净值")
ax.set_ylabel("NAV")
plt.show()


## 4) 多因子滚动回归


In [ ]:
factors = ["size", "mom20", "vol20"]
all_dates = sorted(pc["date"].unique())
preds = []

for i in range(12, len(all_dates)):
    train_dates = all_dates[i - 12:i]
    test_date = all_dates[i]

    tr = pc[pc["date"].isin(train_dates)]
    te = pc[pc["date"] == test_date].copy()
    if len(tr) < 50 or len(te) < 20:
        continue

    X = tr[factors].values
    y = tr["future_ret"].values
    beta = np.linalg.lstsq(np.c_[np.ones(len(X)), X], y, rcond=None)[0]

    te["pred_ret"] = np.c_[np.ones(len(te)), te[factors].values] @ beta
    preds.append(te[["date", "asset", "pred_ret", "future_ret"]])

pred = pd.concat(preds, ignore_index=True)
pred["pred_q"] = pred.groupby("date")["pred_ret"].transform(lambda x: pd.qcut(x.rank(method="first"), 5, labels=False))

mf = pred.groupby(["date", "pred_q"])["future_ret"].mean().unstack()[4]
mf_nav = (1 + mf).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
(100 * mf_nav / mf_nav.iloc[0]).plot(ax=ax, title="C05 在线版多因子策略净值")
ax.set_ylabel("Index=100")
plt.show()

report = pd.DataFrame({
    "rank_ic_mean": [rank_ic.mean()],
    "rank_ic_ir": [rank_ic.mean() / (rank_ic.std() + 1e-12)],
    "single_factor_ann_ret": [ls.mean() * 12],
    "multi_factor_ann_ret": [mf.mean() * 12],
}).round(4)
report
